In [5]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import json

from IPython.display import Markdown, display, update_display

In [6]:
load_dotenv(override=True)
ollama_url = os.getenv("OLLAMA_BASE_URL")
openai = OpenAI(base_url=ollama_url, api_key="")  # No API key needed for Ollama local server

In [38]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
Do not browse the link.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links[10:])

    return user_prompt

In [ ]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model="gemma3:4b",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    print(result)
    links = json.loads(result)
    return links
    

In [40]:
releventlinks = select_relevant_links("https://timesofindia.indiatimes.com")
releventlinks

{}

In [41]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [42]:
brochure_user_prompt = """You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
""" + """Website Content:
{website_content}
Relevant Links:
{relevant_links}
"""

In [43]:
client = OpenAI(base_url=ollama_url, api_key="")

def create_sales_brochure(website_content, relevant_links):
    stream = client.chat.completions.create(
        model="gemma3:4b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": brochure_user_prompt.format(website_content=website_content, relevant_links=relevant_links)}
        ],
        stream=True
    )
    return stream

In [44]:
responseStream = create_sales_brochure(fetch_website_contents("https://google.com"), releventlinks)
display(Markdown("### Sales Brochure:\n"))
responseText = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in responseStream:
    responseText += chunk.choices[0].delta.get("content", "")
    display_handle.update_display(Markdown(responseText), display_id = display_handle.display_id)
    

KeyError: 'company_name'